# Colab용 영화 리뷰 분석기 (Ollama + Gemma4)
이 노트북은 구글 드라이브를 마운트하고 Ollama를 로컬에 설치한 뒤, 리뷰 데이터를 분석하는 과정을 담고 있습니다.

In [1]:
import os
import shutil
from google.colab import drive

mountpoint = '/content/drive'
# 경로가 존재하지만 마운트된 상태가 아닐 경우 기존 로컬 폴더 삭제
if os.path.exists(mountpoint) and not os.path.ismount(mountpoint):
    shutil.rmtree(mountpoint)

# 구글 드라이브 마운트
drive.mount(mountpoint, force_remount=True)


Mounted at /content/drive


In [2]:
# 2. Ollama 설치 및 백그라운드 실행, 모델 다운로드
!apt-get install zstd -y
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time

# 서버 실행
process = subprocess.Popen(['ollama', 'serve'])
time.sleep(3)

# 파이썬용 ollama 패키지 설치
!pip install ollama

# 모델 다운로드 (gemma4 31b)
!ollama pull gemma4:31b

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.



In [3]:
import subprocess

try:
    gpu_status = subprocess.check_output(['nvidia-smi']).decode('utf-8')
    print("✅ GPU가 감지되었습니다:")
    print(gpu_status)
except Exception:
    print("❌ GPU가 감지되지 않았습니다. 런타임 유형이 GPU로 설정되었는지 확인해 주세요.")

✅ GPU가 감지되었습니다:
Thu Apr 30 06:04:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             40W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-------------------------------

### GPU 인식 후 Ollama 재설치 및 모델 로드
GPU 런타임으로 변경한 후에는 아래 코드를 다시 실행하여 GPU 가속이 적용된 상태로 Ollama를 실행해야 합니다.

In [4]:
# 기존 프로세스 종료 (필요시)
!pkill ollama

# GPU 지원을 위한 드라이버 관련 패키지 설치
!apt-get install -y pciutils lshw

# Ollama 재설치 및 서버 실행
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time

process = subprocess.Popen(['ollama', 'serve'])
time.sleep(5)

# 💡 초고속 경량 모델 다운로드
!ollama pull llama3.2

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
lshw is already the newest version (02.19.git.2021.06.19.996aaad9c7-2build1).
pciutils is already the newest version (1:3.7.0-6).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.



In [5]:
# 3. 라이브러리 임포트 및 경로 설정
import json
import os
import time
import re
import ollama

# ጰ️ 코랩 환경에서의 프로젝트 루트 경로 설정
ROOT_DIR = "/content/drive/MyDrive/MovieReview/"
DATA_DIR = os.path.join(ROOT_DIR, "data")

# 💡 5시간 내 처리를 위한 초경량/초고속 모델로 변경 (3B 파라미터)
LOCAL_MODEL = "llama3.2"

# 🚀 테스트 모드 스위치
IS_TEST_MODE = False  # True: 일부 리뷰만 분석 / False: 전체 분석
TEST_LIMIT = 100

print(f"데이터 경로 설정 완료: {DATA_DIR}")
print(f"사용 모델 변경 완료: {LOCAL_MODEL}")

데이터 경로 설정 완료: /content/drive/MyDrive/MovieReview/data
사용 모델 변경 완료: llama3.2


In [6]:
# 4. 태그 룰 & 분류 함수 정의
TAG_RULES = {
    "연기좋음": ["열연", "명연기", "연기력", "인생 캐릭터", "소름", "연기 잘"],
    "연기나쁨": ["연기력 논란", "발연기", "어색", "몰입 방해", "연기 못"],
    "연출좋음": ["영상미", "미장센", "감각적", "탁월한 연출", "연출력"],
    "연출나쁨": ["조잡", "촌스러운", "연출력 부족", "엉성한 연출"],
    "서사좋음": ["탄탄한", "짜임새", "명작", "완벽한 서사", "개연성"],
    "서사나쁨": ["지루", "뻔한", "용두사미", "허술", "개연성 부족"],
    "비주얼좋음": ["비주얼", "영상미", "눈이 즐거운", "그래픽", "훌륭한 CG"],
    "비주얼나쁨": ["허접한 CG", "촌스러운 비주얼", "어색한 그래픽"],
    "음악좋음": ["ost", "음악", "사운드트랙", "배경음악", "음악이 좋"],
    "음악나쁨": ["음악이 안 어울", "시끄러운", "음악이 별로"],
    "분위기좋음": ["분위기 있", "감성적인", "매력적인 분위기", "무드"],
    "분위기나쁨": ["산만한 분위기", "분위기 깨는", "칙칙한"],
    "주의사항": ["노출", "가족", "민망", "야한", "잔인", "공포", "폭력"],
    "전체긍정": ["최고", "추천", "재밌", "훌륭", "완벽", "인생 영화", "강추"],
    "전체부정": ["최악", "비추", "노잼", "별로", "시간 아깝", "실망", "돈 아깝"]
}

def classify_with_rule_base(content):
    found_tags = []
    extracted_keywords = []
    for tag_name, keywords in TAG_RULES.items():
        matched = [k for k in keywords if k in content]
        if matched:
            found_tags.append(tag_name)
            extracted_keywords.extend(matched)
    if not found_tags: found_tags.append("일반감상")
    return found_tags, list(set(extracted_keywords))

def classify_single_review_with_local_llm(review, max_retries=2):
    prompt = f"""
    당신은 영화 리뷰 분석 전문가입니다. 아래 리뷰를 읽고 범주에서 알맞은 태그를 선택하고, 핵심 키워드를 추출하세요.

    [범주]: 전체긍정, 전체부정, 전체복합, 연기좋음, 연기나쁨, 연출좋음, 연출나쁨, 서사좋음, 서사나쁨, 비주얼좋음, 비주얼나쁨, 음악좋음, 분위기가벼움, 분위기무거움, 고증좋음, 고증나쁨, 주의사항, TMI, 장르특성

    [지시사항]
    - 반드시 아래 JSON 형식으로만 대답하세요. 다른 인사말이나 설명은 절대 넣지 마세요.
    - content_character 에는 범주에 해당하는 태그만 넣으세요.
    - search_keywords 에는 리뷰 본문에서 추출한 핵심 단어를 넣으세요.

    ```json
    {{
        "content_character": ["태그1", "태그2"],
        "search_keywords": ["키워드1", "키워드2"]
    }}
    ```

    [리뷰 본문]:
    "{review['content']}"
    """

    import time
    for attempt in range(max_retries):
        try:
            # format='json' 제거하여 모델이 강박적으로 빈 {} 를 반환하는 현상 방지
            response = ollama.chat(model=LOCAL_MODEL, messages=[
                {'role': 'user', 'content': prompt},
            ])

            response_text = response['message']['content']

            # 정규식으로 JSON 부분만 추출
            json_match = re.search(r'\{.*?\}', response_text, re.DOTALL)
            if not json_match:
                raise ValueError("JSON 형식을 찾을 수 없습니다.")

            json_text = json_match.group(0)
            res = json.loads(json_text)

            content_chars = res.get("content_character", [])
            keywords = res.get("search_keywords", [])

            if not content_chars or not keywords:
                raise ValueError("태그나 키워드가 비어있습니다.")

            return res
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(0.5)
            else:
                print(f"⚠️ 분석 실패 (Fallback 전환): {e}")
                return None

In [7]:
# 5. 메인 실행 블록
from tqdm.notebook import tqdm
import os
import json

input_path = os.path.join(DATA_DIR, "cleaned_total_reviews.json")

if IS_TEST_MODE:
    output_path = os.path.join(DATA_DIR, "test_tagged_reviews.json")
else:
    output_path = os.path.join(DATA_DIR, "tagged_reviews.json")

if not os.path.exists(input_path):
    print(f"❌ '{input_path}' 파일이 없습니다. 경로를 확인해 주세요.")
else:
    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if os.path.exists(output_path):
        with open(output_path, "r", encoding="utf-8") as f:
            processed_data = json.load(f)
        print("📝 기존 작업 결과에서 이어하기를 시작합니다.")

        for m_name, m_data in processed_data.items():
            if m_name in data:
                processed_reviews_dict = {r.get("review_id"): r for r in m_data.get("reviews", [])}
                for i, rev in enumerate(data[m_name].get("reviews", [])):
                    if rev.get("review_id") in processed_reviews_dict:
                        data[m_name]["reviews"][i] = processed_reviews_dict[rev["review_id"]]

    pending_reviews = []
    for m_name, m_data in data.items():
        for rev in m_data.get("reviews", []):
            if rev.get("meta_tags", {}).get("analysis_method") not in ["Local-LLM", "Rule-based (Fallback)"]:
                pending_reviews.append(rev)

    if not pending_reviews:
        print("✅ 모든 리뷰에 대한 분석이 이미 완료되었습니다.")
    else:
        if IS_TEST_MODE:
            pending_reviews = pending_reviews[:TEST_LIMIT]
            print(f"⚠️ [테스트 모드 활성화] 리뷰 수를 {len(pending_reviews)}개로 제한하여 분석합니다.")

        print(f"🚀 총 {len(pending_reviews)}개의 리뷰를 로컬 LLM({LOCAL_MODEL})으로 분석합니다.")

        completed_count = 0

        for rev in tqdm(pending_reviews, desc="리뷰 분석 진행률"):
            res = classify_single_review_with_local_llm(rev)

            if res:
                rev["meta_tags"].update({
                    "content_character": res.get("content_character", []),
                    "search_keywords": res.get("search_keywords", []),
                    "is_tmi": "TMI" in res.get("content_character", []),
                    "has_warning": "주의사항" in res.get("content_character", []),
                    "analysis_method": "Local-LLM"
                })
            else:
                # 실패 시 Rule-based Fallback
                tags, keywords = classify_with_rule_base(rev["content"])
                rev["meta_tags"].update({
                    "content_character": tags,
                    "search_keywords": keywords,
                    "analysis_method": "Rule-based (Fallback)"
                })

            completed_count += 1
            # 100개 완료될 때마다 중간 저장 (저장 빈도 조정)
            if completed_count % 100 == 0:
                filtered_data = {}
                for m_name, m_data in data.items():
                    processed_reviews = [r for r in m_data.get("reviews", []) if r.get("meta_tags", {}).get("analysis_method") in ("Local-LLM", "Rule-based (Fallback)")]
                    if processed_reviews:
                        filtered_data[m_name] = {"movie_metadata": m_data.get("movie_metadata", {}), "reviews": processed_reviews}
                with open(output_path, "w", encoding="utf-8") as f:
                    json.dump(filtered_data, f, ensure_ascii=False, indent=4)

        # 최종 저장
        filtered_data = {}
        for m_name, m_data in data.items():
            processed_reviews = [r for r in m_data.get("reviews", []) if r.get("meta_tags", {}).get("analysis_method") in ("Local-LLM", "Rule-based (Fallback)")]
            if processed_reviews:
                filtered_data[m_name] = {"movie_metadata": m_data.get("movie_metadata", {}), "reviews": processed_reviews}

        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(filtered_data, f, ensure_ascii=False, indent=4)

        print(f"\n✨ 로컬 LLM 분석 완료! 결과 저장: {output_path}")

📝 기존 작업 결과에서 이어하기를 시작합니다.
🚀 총 7985개의 리뷰를 로컬 LLM(llama3.2)으로 분석합니다.


리뷰 분석 진행률:   0%|          | 0/7985 [00:00<?, ?it/s]

⚠️ 분석 실패 (Fallback 전환): JSON 형식을 찾을 수 없습니다.
⚠️ 분석 실패 (Fallback 전환): JSON 형식을 찾을 수 없습니다.
⚠️ 분석 실패 (Fallback 전환): JSON 형식을 찾을 수 없습니다.
⚠️ 분석 실패 (Fallback 전환): JSON 형식을 찾을 수 없습니다.
⚠️ 분석 실패 (Fallback 전환): 태그나 키워드가 비어있습니다.
⚠️ 분석 실패 (Fallback 전환): Expecting value: line 7 column 12 (char 89)
⚠️ 분석 실패 (Fallback 전환): JSON 형식을 찾을 수 없습니다.
⚠️ 분석 실패 (Fallback 전환): Expecting ',' delimiter: line 19 column 4 (char 336)
⚠️ 분석 실패 (Fallback 전환): JSON 형식을 찾을 수 없습니다.
⚠️ 분석 실패 (Fallback 전환): 태그나 키워드가 비어있습니다.
⚠️ 분석 실패 (Fallback 전환): 태그나 키워드가 비어있습니다.
⚠️ 분석 실패 (Fallback 전환): JSON 형식을 찾을 수 없습니다.
⚠️ 분석 실패 (Fallback 전환): 태그나 키워드가 비어있습니다.
⚠️ 분석 실패 (Fallback 전환): Expecting ',' delimiter: line 8 column 4 (char 125)
⚠️ 분석 실패 (Fallback 전환): Expecting ',' delimiter: line 18 column 4 (char 278)
⚠️ 분석 실패 (Fallback 전환): Expecting ',' delimiter: line 5 column 4 (char 65)
⚠️ 분석 실패 (Fallback 전환): Expecting ':' delimiter: line 3 column 17 (char 43)
⚠️ 분석 실패 (Fallback 전환): Expecting ',' delimiter: line 12 column 4 (char 206)
⚠️